In [1]:
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass


Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/M3_colab
%pip install -q xgboost lightgbm shap

/content/drive/MyDrive/M3_colab


In [ ]:
import os

# Fix the missing 'import os' in the build_training_table.py file if it's missing
script_path = "src/build_training_table.py"
with open(script_path, "r", encoding="utf-8") as f:
    content = f.read()

if "import os" not in content:
    print(f"Adding 'import os' to {script_path}...")
    content = "import os\n" + content
    with open(script_path, "w", encoding="utf-8") as f:
        f.write(content)

os.environ["M3_SAMPLES_DIR"] = "/content/drive/MyDrive/M3_colab/src/samples"
!python src/build_training_table.py

Ratnapura: M3_Ratnapura_TrainingSamples_TS_2025-10-01_2025-11-06.csv -> 4547 pixels, 46 fields
Colombo: M3_Colombo_TrainingSamples_TS_2025-10-01_2025-11-06.csv -> 3516 pixels, 24 fields
Gampaha: M3_Gampaha_TrainingSamples_TS_2025-10-01_2025-11-06.csv -> 5480 pixels, 30 fields

Classes: ['Other', 'Paddy', 'Tea']

Fields per district x class:
crop_type  Other  Paddy  Tea
district                    
Colombo       10     12    2
Gampaha       18     12    0
Ratnapura     10     10   26

Pixels per district x class:
crop_type  Other  Paddy   Tea
district                     
Colombo     1226   2263    27
Gampaha     1222   4258     0
Ratnapura   1115   2261  1171

Pixels per field (min / median / max): [4.0, 76.5, 1047.0]
Share of pixels by fill_level: {0.0: 0.914, 2.0: 0.063, 1.0: 0.023}

Missing share per district, 10 worst time-series columns:
district      Colombo  Gampaha  Ratnapura  worst
NDVI_2025_09    0.484    0.191      0.101  0.484
NDWI_2025_09    0.484    0.191      0.101  0.48

In [ ]:
!python src/train_evaluate.py

{'base': 15, 'base+timeseries': 41}
base             Random Forest  macro-F1 per fold: [0.98  0.999 0.957 0.782 1.   ]
base             XGBoost        macro-F1 per fold: [0.985 0.998 0.954 0.786 0.97 ]
base             LightGBM       macro-F1 per fold: [0.981 0.999 0.956 0.784 0.997]
base             SVM (RBF)      macro-F1 per fold: [0.978 0.99  0.91  0.78  0.994]
base+timeseries  Random Forest  macro-F1 per fold: [0.974 0.999 0.958 0.792 1.   ]
base+timeseries  XGBoost        macro-F1 per fold: [0.978 0.999 0.96  0.734 0.933]
base+timeseries  LightGBM       macro-F1 per fold: [0.973 0.999 0.96  0.71  0.999]
base+timeseries  SVM (RBF)      macro-F1 per fold: [0.978 0.998 0.952 0.81  0.957]

     feature_set         model  overall_accuracy  macro_f1  kappa  fold_f1_mean  fold_f1_std  field_accuracy  field_macro_f1  F1_Other  F1_Paddy  F1_Tea
base+timeseries Random Forest            0.9600    0.9211 0.9211        0.9446       0.0778            0.92          0.9187    0.9263    0.9907  0

In [ ]:
!python src/explain_shap.py

               Other   Paddy     Tea  All classes
elevation     0.1677  0.0138  0.1797       0.1204
NDVI          0.0643  0.0957  0.0314       0.0638
NDWI          0.0336  0.0548  0.0212       0.0365
slope         0.0342  0.0048  0.0390       0.0260
B7            0.0264  0.0382  0.0124       0.0257
B8            0.0215  0.0329  0.0168       0.0237
B8A           0.0161  0.0273  0.0149       0.0195
B6            0.0156  0.0254  0.0139       0.0183
NDWI_mean     0.0134  0.0236  0.0126       0.0165
NDVI_2025_09  0.0193  0.0018  0.0209       0.0140
NDVI_p10      0.0114  0.0205  0.0095       0.0138
B11           0.0160  0.0041  0.0200       0.0134
NDVI_mean     0.0098  0.0199  0.0103       0.0133
NDWI_2025_09  0.0164  0.0022  0.0184       0.0123
NDVI_min      0.0134  0.0158  0.0029       0.0107


In [ ]:
!python src/evaluate_extra.py

13543 rows | 100 fields | classes ['Other', 'Paddy', 'Tea']

[1] Feature-set ablations (repeated field-grouped CV)
  full (15 base + 26 TS)     macro-F1 0.932 +- 0.019 | Tea F1 0.881
  base only (15)             macro-F1 0.924 +- 0.022 | Tea F1 0.869
  no terrain                 macro-F1 0.844 +- 0.005 | Tea F1 0.688
  no September               macro-F1 0.934 +- 0.019 | Tea F1 0.883
  no terrain, no Sept        macro-F1 0.808 +- 0.017 | Tea F1 0.606
  spectral + indices only    macro-F1 0.785 +- 0.026 | Tea F1 0.589
  time series only           macro-F1 0.808 +- 0.004 | Tea F1 0.626

[2] Per-district results, curves and field-level view (full feature set)
 district  pixels  fields         classes  accuracy  macro_f1_present_classes  recall_Other  recall_Paddy  recall_Tea
  Colombo    3516      24 Other/Paddy/Tea    0.9733                    0.6490        0.9625        0.9907      0.0000
  Gampaha    5480      30     Other/Paddy    0.9951                    0.9929        0.9845        

In [ ]:
!python src/explain_extra.py

Explaining: Random Forest | base+timeseries (41 features)
SHAP computed for 1500 pixels
Permutation importance on held-out fields (a few minutes)...

Done. Files are in /content/drive/MyDrive/M3_colab/outputs/results/extra_xai

Top features by SHAP:
                Other   Paddy     Tea  All classes
elevation     0.1670  0.0137  0.1791       0.1199
NDVI          0.0645  0.0959  0.0315       0.0639
NDWI          0.0337  0.0549  0.0213       0.0366
slope         0.0344  0.0048  0.0392       0.0262
B7            0.0262  0.0379  0.0124       0.0255
B8            0.0214  0.0327  0.0167       0.0236
B8A           0.0160  0.0271  0.0148       0.0193
B6            0.0155  0.0251  0.0138       0.0181
NDWI_mean     0.0135  0.0237  0.0126       0.0166
NDVI_2025_09  0.0194  0.0018  0.0209       0.0141

Feature-group share of SHAP (%):
                      Other  Paddy   Tea  All classes
Spectral bands        21.4   31.2  20.0         23.8
Indices (composite)   16.4   32.0  10.1         18.9
Terra

In [5]:
%pip install -q rasterio requests
import os
os.environ["M3_RASTER_DIR"] = "/content/drive/MyDrive/M3_colab/rasters"


In [3]:
import os
os.environ["M3_RASTER_DIR"] = "/content/drive/MyDrive/M3_colab/src/rasters"

In [11]:
!python src/m3_landuse_mask.py


Ratnapura: grid from M3_Ratnapura_FeatureStack_2025-10-01_2025-11-06.tif
Traceback (most recent call last):
  File "/content/drive/MyDrive/M3_colab/src/m3_landuse_mask.py", line 137, in <module>
    results = {d: process(d) for d in DISTRICTS}
                  ~~~~~~~^^^
  File "/content/drive/MyDrive/M3_colab/src/m3_landuse_mask.py", line 100, in process
    with rasterio.open(stack) as s:
         ~~~~~~~~~~~~~^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/rasterio/env.py", line 464, in wrapper
    return f(*args, **kwds)
  File "/usr/local/lib/python3.13/dist-packages/rasterio/__init__.py", line 367, in open
    dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)
  File "rasterio/_base.pyx", line 350, in rasterio._base.DatasetBase.__init__
  File "rasterio/_base.pyx", line 371, in rasterio._base.DatasetBase._set_attrs_from_dataset_handle
    _ = self.meta
  File "rasterio/_base.pyx", line 996, in rasterio._base.DatasetBase.meta

In [ ]:
!python src/m3_final_map.py

Final Random Forest trained on 13543 pixels / 100 fields -> final_rf_base_map.joblib

Ratnapura: M3_Ratnapura_FeatureStack_2025-10-01_2025-11-06.tif
  using the national land-use cultivated mask: M3_Ratnapura_Cultivated.tif (WorldCover and forest mask ignored)
class   pixels  area_ha  share_of_mapped_pct
Other 10284766 101497.7                46.65
Paddy  3085651  30454.6                14.00
  Tea  8674310  85611.7                39.35
  pixels in mask without data: 21322031 (49.17% of the mask)

Colombo: M3_Colombo_FeatureStack_2025-10-01_2025-11-06.tif
  using the national land-use cultivated mask: M3_Colombo_Cultivated.tif (WorldCover and forest mask ignored)
class  pixels  area_ha  share_of_mapped_pct
Other 3863899  38112.0                68.25
Paddy 1713575  16902.4                30.27
  Tea   83998    828.5                 1.48
  pixels in mask without data: 4336058 (43.37% of the mask)

Gampaha: M3_Gampaha_FeatureStack_2025-10-01_2025-11-06.tif
  using the national land-use cu

In [ ]:
!python src/m3_country_map.py

Saved /content/drive/MyDrive/M3_colab/outputs/results/map/SriLanka_3_districts_crop_map.png


In [3]:
import pandas as pd
t = pd.read_csv("outputs/reference_dataset/M3_training_table_all_districts.csv")
print(t[t["crop_type"]=="Paddy"]["NDVI"].describe(percentiles=[.05,.10,.50]))

count    8782.000000
mean        0.273445
std         0.090439
min         0.028497
5%          0.155746
10%         0.180117
50%         0.257003
max         0.780376
Name: NDVI, dtype: float64


In [12]:
os.environ.pop("M3_PADDY_NDVI_MIN", None)
os.environ.pop("M3_USE_RULES", None)
os.environ.pop("M3_PADDY_LU", None)

In [13]:
!python src/m3_final_map.py --skip-train


Ratnapura: M3_Ratnapura_FeatureStack_2025-10-01_2025-11-06.tif
  using the national land-use cultivated mask: M3_Ratnapura_Cultivated.tif (WorldCover and forest mask ignored)
  class rules ON: Paddy only in ['Paddy', 'Paddy-abandoned'] | Tea only in ['Tea']
class   pixels  area_ha  share_of_mapped_pct
Other 18279347 180399.8                82.92
Paddy  1028998  10156.0                 4.67
  Tea  2736382  27008.2                12.41
  pixels in mask without data: 21322031 (49.17% of the mask)

Colombo: M3_Colombo_FeatureStack_2025-10-01_2025-11-06.tif
  using the national land-use cultivated mask: M3_Colombo_Cultivated.tif (WorldCover and forest mask ignored)
  class rules ON: Paddy only in ['Paddy', 'Paddy-abandoned'] | Tea only in ['Tea']
class  pixels  area_ha  share_of_mapped_pct
Other 5416827  53429.7                95.68
Paddy  244122   2408.1                 4.31
  Tea     523      5.2                 0.01
  pixels in mask without data: 4336058 (43.37% of the mask)

Gampaha: M

In [14]:
!python src/m3_country_map.py

Saved /content/drive/MyDrive/M3_colab/outputs/results/map/SriLanka_3_districts_crop_map.png


In [4]:
!python src/m3_area_validation.py

 district  crop  satellite_ha  reference_ha  difference_pct  agreement_pct  meets_80pct
Ratnapura Paddy       10156.0       12269.0           -17.2           82.8         True
Ratnapura   Tea       27008.2       43077.0           -37.3           62.7        False
  Colombo Paddy        2408.1        3422.0           -29.6           70.4        False
  Colombo   Tea           5.2         404.0           -98.7            1.3        False
  Gampaha Paddy        8943.2       10168.0           -12.0           88.0         True
  Gampaha   Tea           0.0         100.0          -100.0            0.0        False

Paddy: mean absolute % error = 19.6% over 3 districts

Tea: mean absolute % error = 78.7% over 3 districts

Saved /content/drive/MyDrive/M3_colab/outputs/results/map/area_validation.csv and area_validation.png
Remember: official figures are sown/planted extents for a season or year; a single pre-flood
image and the land-cover mask will not match them exactly. Explain differences, 

In [5]:
!python src/m3_damage_map.py


Ratnapura
  pre : M3_Ratnapura_FeatureStack_2025-10-01_2025-11-06.tif
  post: M3_Ratnapura_PostFlood_2025-12-10_2026-01-16.tif
  S1  : M3_Ratnapura_S1Flood.tif
  usable pixels (crop map & both dates clear): 22,035,278 of 22,044,727 mapped
 crop  assessed_ha  damaged_ha  damage_pct_of_assessed  baseline_change_median
Other     180319.4      9118.5                    5.06                  -0.010
Paddy      10151.6       198.8                    1.96                   0.256
  Tea      26999.7      2080.9                    7.71                   0.000

Colombo
  pre : M3_Colombo_FeatureStack_2025-10-01_2025-11-06.tif
  post: M3_Colombo_PostFlood_2025-12-10_2026-01-16.tif
  S1  : M3_Colombo_S1Flood.tif
  usable pixels (crop map & both dates clear): 5,656,799 of 5,661,472 mapped
 crop  assessed_ha  damaged_ha  damage_pct_of_assessed  baseline_change_median
Other      53386.9      3382.4                    6.34                  -0.017
Paddy       2404.7       213.5                    8.88  

In [4]:
!python src/m3_export_web.py

Writing to /content/drive/MyDrive/M3_colab/web/data

Ratnapura
/content/drive/MyDrive/M3_colab/src/m3_export_web.py:123: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(rgba, "RGBA").save(f, optimize=True)
/content/drive/MyDrive/M3_colab/src/m3_export_web.py:123: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(rgba, "RGBA").save(f, optimize=True)
/content/drive/MyDrive/M3_colab/src/m3_export_web.py:123: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(rgba, "RGBA").save(f, optimize=True)
/content/drive/MyDrive/M3_colab/src/m3_export_web.py:123: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(rgba, "RGBA").save(f, optimize=True)
/content/drive/MyDrive/M3_colab/src/m3_export_web.py:123: DeprecationWarning: 'mode' parameter is

In [5]:
import rasterio
with rasterio.open("/content/drive/MyDrive/M3_colab/src/rasters/Gampaha/M3_Gampaha_PostFlood_2025-12-10_2026-01-16.tif") as s:
    print(s.count, s.descriptions, s.res)

3 ('NDVI_post', 'NDWI_post', 'post_fill') (8.983152841195215e-05, 8.983152841195215e-05)


In [4]:
!python src/m3_damage_map_v2.py


Ratnapura
  pre : M3_Ratnapura_FeatureStack_2025-10-01_2025-11-06.tif
  post: M3_Ratnapura_PostFlood_2025-12-10_2026-01-16.tif
  S1  : M3_Ratnapura_S1Flood.tif
  base: M3_Ratnapura_SeasonalBaseline.tif
  pixels with a seasonal baseline: 97.3% of usable
  usable pixels (crop map & both dates clear): 22,035,278 of 22,044,727 mapped
 crop  assessed_ha  damaged_ha  damage_pct_of_assessed  baseline_sigma
Other     180319.4     14755.3                    8.18           0.078
Paddy      10151.6       192.3                    1.89           0.231
  Tea      26999.7      2263.1                    8.38           0.081
  Damage rate inside vs outside Sentinel-1 flood:
  district  crop  S1_flooded_ha  damaged_in_flood_pct  damaged_outside_flood_pct  times_higher  share_of_damage_inside_flood_pct
Ratnapura Other          106.8                  31.6                        8.2           3.9                               0.2
Ratnapura Paddy          383.5                   4.6                        